# 🌐 Phase 1 — Multilingual Translator: Dataset Build + Hybrid Model Training

**Architecture:** XLM-RoBERTa encoder (first 6 layers frozen) + Custom 6-layer Transformer decoder built from scratch.

**Languages covered:**
- 🇮🇳 Indian: Hindi, Bengali, Tamil, Telugu, Marathi, Gujarati, Kannada, Malayalam, Punjabi, Urdu
- 🇪🇺 European: French, German, Spanish, Italian, Portuguese, Dutch, Polish, Romanian, Swedish, Czech
- 🀄 CJK: Chinese (Simplified), Japanese, Korean
- 🇬🇧 English (pivot)

**Corpus sources:** `Helsinki-NLP/opus-100`, `opus_books`, `Helsinki-NLP/tatoeba_mt`

**Hardware:** 2× NVIDIA T4 (16 GB each)

---

## Workflow
1. Install dependencies  
2. Set secrets  
3. Build & upload the dataset  
4. Train the hybrid model  
5. Inspect metrics  

## Step 1 — Install Dependencies

In [ ]:
import subprocess, sys

def pip_install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade'] + packages
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-3000:])
    else:
        print(f'Installed: {" ".join(packages)}')

# Core packages
pip_install([
    'torch==2.1.2',
    'transformers==4.38.2',
    'datasets==2.18.0',
    'tokenizers==0.15.2',
    'huggingface-hub==0.21.4',
    'accelerate==0.27.2',
    'evaluate==0.4.1',
])

pip_install([
    'pandas==2.2.1',
    'pyarrow==15.0.2',
    'sacrebleu==2.4.0',
    'langdetect==1.0.9',
    'tqdm==4.66.2',
    'scikit-learn==1.4.1.post1',
    'matplotlib==3.8.3',
    'pynvml==11.5.0',
])

print('\n✅ All packages installed')

## Step 2 — Configure Secrets & Environment

In [ ]:
import os

# ── Read from Kaggle secrets ───────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN    = secrets.get_secret('HF_TOKEN')
    HF_USERNAME = secrets.get_secret('HF_USERNAME')
    print(f'✅ Secrets loaded. HF_USERNAME={HF_USERNAME}')
except Exception as e:
    print(f'⚠️  Kaggle secrets not available ({e}). Set env vars manually below.')
    HF_TOKEN    = ''   # ← paste your token here if running locally
    HF_USERNAME = ''   # ← paste your username here

os.environ['HF_TOKEN']    = HF_TOKEN
os.environ['HF_USERNAME'] = HF_USERNAME
os.environ['OUTPUT_DIR']  = '/kaggle/working/multilingual_dataset'
os.environ['DATA_DIR']    = '/kaggle/working/multilingual_dataset'
os.environ['CHECKPOINT_DIR'] = '/kaggle/working/checkpoints'

print(f'OUTPUT_DIR   = {os.environ["OUTPUT_DIR"]}')
print(f'CHECKPOINT_DIR = {os.environ["CHECKPOINT_DIR"]}')

## Step 3 — GPU & System Info

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'GPU count       : {n_gpus}')
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — training will be slow on CPU.')

## Step 4 — Copy Source Scripts to Working Dir

In [ ]:
import shutil, pathlib

# When running directly on Kaggle upload the .py files as a dataset or paste inline.
# Here we write them out from inline content so the notebook is self-contained.

WORK = pathlib.Path('/kaggle/working')

# Check if scripts were uploaded as dataset
DATASET_INPUT = pathlib.Path('/kaggle/input')
candidates = list(DATASET_INPUT.rglob('dataset_builder.py'))

if candidates:
    src = candidates[0].parent
    for f in src.glob('*.py'):
        shutil.copy(f, WORK / f.name)
    print(f'Copied scripts from dataset: {src}')
else:
    # Scripts should be in the same directory as this notebook
    notebook_dir = pathlib.Path('.').resolve()
    for f in notebook_dir.glob('*.py'):
        shutil.copy(f, WORK / f.name)
        print(f'Copied {f.name} → {WORK / f.name}')

print('\nFiles in /kaggle/working:')
for p in sorted(WORK.iterdir()):
    print(f'  {p.name}')

## Step 5 — Build the Multilingual Dataset

Downloads corpora from HuggingFace Hub, cleans, deduplicates, creates 90/5/5 splits, saves sharded Parquet files, and pushes to your HuggingFace dataset repo.

In [ ]:
import subprocess, sys, time

print('=' * 60)
print('RUNNING: dataset_builder.py')
print('=' * 60)

t0 = time.time()
result = subprocess.run(
    [sys.executable, '/kaggle/working/dataset_builder.py'],
    env=os.environ.copy(),
    text=True,
    capture_output=False,   # stream output directly
)
elapsed = time.time() - t0
print(f'\n⏱️  Completed in {elapsed/60:.1f} min  (exit code {result.returncode})')
if result.returncode != 0:
    print('❌ dataset_builder.py failed — check output above.')
else:
    print('✅ Dataset build complete!')

### Dataset Statistics

In [ ]:
import json, pathlib
import pandas as pd
import matplotlib.pyplot as plt

stats_path = pathlib.Path('/kaggle/working/multilingual_dataset/dataset_statistics.json')
if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)

    print(f"Total rows   : {stats['total_rows']:,}")
    print(f"Train rows   : {stats['train_rows']:,}")
    print(f"Val rows     : {stats['val_rows']:,}")
    print(f"Test rows    : {stats['test_rows']:,}")
    print(f"Elapsed (s)  : {stats.get('elapsed_seconds', 'N/A')}")

    # Language pair bar chart
    pairs = stats.get('language_pairs', {})
    if pairs:
        df_pairs = pd.Series(pairs).sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(14, 5))
        df_pairs.plot(kind='bar', ax=ax, color='steelblue')
        ax.set_title('Rows per Language Pair')
        ax.set_ylabel('Count')
        ax.set_xlabel('Language Pair')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # Domain distribution
    domains = stats.get('domain_distribution', {})
    if domains:
        df_dom = pd.Series(domains)
        fig2, ax2 = plt.subplots(figsize=(6, 4))
        df_dom.plot(kind='pie', ax=ax2, autopct='%1.1f%%')
        ax2.set_title('Domain Distribution')
        ax2.set_ylabel('')
        plt.tight_layout()
        plt.show()
else:
    print('⚠️  Statistics file not found — dataset_builder.py may not have run yet.')

## Step 6 — Train the Hybrid Model

Trains the Custom XLM-RoBERTa + 6-Layer Decoder model with:
- Mixed precision (fp16)
- Gradient checkpointing
- Label-smoothing cross-entropy
- Cosine LR schedule with warmup
- BLEU evaluation on val set
- Early stopping (patience = 3)
- Checkpoint push to HuggingFace Hub every 5000 steps

In [ ]:
import subprocess, sys, time, os

print('=' * 60)
print('RUNNING: train_hybrid_model.py')
print('=' * 60)

t0 = time.time()
result = subprocess.run(
    [sys.executable, '/kaggle/working/train_hybrid_model.py'],
    env=os.environ.copy(),
    text=True,
    capture_output=False,
)
elapsed = time.time() - t0
print(f'\n⏱️  Training finished in {elapsed/3600:.2f} hours  (exit code {result.returncode})')
if result.returncode != 0:
    print('❌ Training script failed — check output above.')
else:
    print('✅ Training complete!')

## Step 7 — Training Metrics Visualisation

In [ ]:
import json, pathlib
import pandas as pd
import matplotlib.pyplot as plt

log_path = pathlib.Path('/kaggle/working/checkpoints/training_log.jsonl')

if log_path.exists():
    records = []
    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    df_log = pd.DataFrame(records)
    print(f'Total log entries: {len(df_log)}')
    print(df_log.tail(10).to_string(index=False))

    # ── Training loss ──────────────────────────────────────────────────────
    train_df = df_log[df_log.get('type', pd.NA).isna() & df_log['train_loss'].notna()] \
               if 'train_loss' in df_log.columns else pd.DataFrame()
    eval_df  = df_log[df_log.get('type', '') == 'eval'] \
               if 'type' in df_log.columns else pd.DataFrame()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if not train_df.empty and 'train_loss' in train_df.columns:
        axes[0].plot(train_df['step'], train_df['train_loss'], color='royalblue', linewidth=0.8)
        axes[0].set_title('Training Loss')
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('Loss')
        axes[0].grid(True, alpha=0.3)

    if not eval_df.empty and 'val_loss' in eval_df.columns:
        axes[1].plot(eval_df['step'], eval_df['val_loss'], color='tomato', marker='o', linewidth=1.5)
        axes[1].set_title('Validation Loss')
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('Val Loss')
        axes[1].grid(True, alpha=0.3)

    if not eval_df.empty and 'bleu' in eval_df.columns:
        axes[2].plot(eval_df['step'], eval_df['bleu'], color='seagreen', marker='s', linewidth=1.5)
        axes[2].set_title('BLEU Score (Val)')
        axes[2].set_xlabel('Step')
        axes[2].set_ylabel('BLEU')
        axes[2].grid(True, alpha=0.3)

    plt.suptitle('custom-hybrid-v1 Training Metrics', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_metrics.png', dpi=150)
    plt.show()
    print('\nPlot saved → /kaggle/working/training_metrics.png')
else:
    print(f'⚠️  Log file not found at {log_path}')

## Step 8 — List Saved Checkpoints

In [ ]:
import pathlib

ckpt_dir = pathlib.Path('/kaggle/working/checkpoints')
checkpoints = sorted(ckpt_dir.glob('checkpoint_step_*.pt'))

if checkpoints:
    print(f'Found {len(checkpoints)} checkpoint(s):')
    for c in checkpoints:
        size_mb = c.stat().st_size / 1e6
        print(f'  {c.name}  ({size_mb:.1f} MB)')
else:
    print('⚠️  No checkpoints found.')

---
## ✅ Phase 1 Complete

| Deliverable | Location |
|---|---|
| Multilingual dataset (Parquet) | `/kaggle/working/multilingual_dataset/` |
| Dataset statistics | `/kaggle/working/multilingual_dataset/dataset_statistics.json` |
| Model checkpoints | `/kaggle/working/checkpoints/` |
| Training log | `/kaggle/working/checkpoints/training_log.jsonl` |
| Metrics plot | `/kaggle/working/training_metrics.png` |
| HuggingFace dataset repo | `{HF_USERNAME}/multilingual-translation-data` |
| HuggingFace model repo | `{HF_USERNAME}/custom-hybrid-v1` |

**Next:** Phase 2 — Fine-tune for specific language pairs and distil to mobile-optimised models.